# Connect adapter sources

Use the Python adapter definitions from the examples with a local ITCH file, a Nasdaq sample URL, and the Bitfinex public feed.

`CustomAdapter` compiles the Python declaration before the source starts. The build notebooks show the complete declarations and reproducible raw-input examples: [ITCH](custom_adapter_itch.ipynb), [Kraken](custom_adapter_kraken.ipynb), [Bitfinex](custom_adapter_bitfinex.ipynb), and [order API](custom_adapter_order_api.ipynb).

In [1]:
from pathlib import Path
from time import perf_counter
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "rust/crates/lobo_replay").is_dir())
sys.path.insert(0, str(root / "notebooks"))
from adapter_examples import wait_for_book
import pandas as pd
from IPython.display import display, HTML
from lobo.server import server_context
sys.path.insert(0, str(root / "python"))
from examples.itch import adapter as itch
from examples.bitfinex import adapter as bitfinex
from lobo.replay.adapters import CustomAdapter
from lobo.replay.adapters import models as lm

Construction timings include preparation and may reuse cached code; they are not throughput benchmarks. Run cells individually to explore the terminal before cleanup. Run All closes each session. Online sections require internet access.

## Local ITCH file

In [2]:
definition = itch.protocol()
source = lm.Source.file(root / "data/NASDAQ/01302020.NASDAQ_ITCH50")
started = perf_counter()
adapter = CustomAdapter(
    definition, source, name='ITCH', symbol='AAPL', scope=['AAPL'],
    mode='replay', level="l3", timezone='America/New_York',
)
print(f"Construction (includes compilation or cache reuse): {(perf_counter() - started) * 1000:.3f} ms")
terminal = server_context(adapters=[adapter], port=0)
display(HTML(f'<a href="{terminal.url}" target="_blank">Open order-book terminal</a>'))

Construction (includes compilation or cache reuse): 28.978 ms


In [3]:
display(pd.DataFrame(wait_for_book(adapter, 'AAPL')).head(12))
adapter.status()

,hidden,orders,price,quantity,side
0,0,1,3213100,100,buy
1,0,1,3212000,500,buy
2,0,1,3211500,325,buy
3,0,1,3211000,500,buy
4,0,1,3210100,30,buy
5,0,4,3210000,86,buy
6,0,3,3209000,2021,buy
7,0,1,3207800,100,buy
8,0,1,3202000,525,buy
9,0,2,3201700,130,buy


{'books': 1,
 'bytes': 54534127,
 'checksum_checks': 0,
 'checksum_failures': 0,
 'clock_ns': 17326991435174,
 'complete': False,
 'messages': 1870456,
 'symbol': 'AAPL',
 'synchronized': True}

Close this session before starting the next source.

In [4]:
terminal.close()

## Nasdaq sample session

In [5]:
definition = itch.protocol()
source = lm.Source.http("https://emi.nasdaq.com/ITCH/Nasdaq%20ITCH/01302020.NASDAQ_ITCH50.gz")
started = perf_counter()
adapter = CustomAdapter(
    definition, source, name='Nasdaq sample', symbol='AAPL', scope=['AAPL'],
    mode='replay', level="l3", timezone='America/New_York',
)
print(f"Construction (includes compilation or cache reuse): {(perf_counter() - started) * 1000:.3f} ms")
terminal = server_context(adapters=[adapter], port=0)
display(HTML(f'<a href="{terminal.url}" target="_blank">Open order-book terminal</a>'))

Construction (includes compilation or cache reuse): 0.313 ms


In [6]:
display(pd.DataFrame(wait_for_book(adapter, 'AAPL')).head(12))
adapter.status()

,hidden,orders,price,quantity,side
0,0,1,3210000,5,buy
1,0,1,3208000,300,buy
2,0,1,3206800,30,buy
3,0,3,3200000,133,buy
4,0,1,3197500,25,buy
5,0,1,3191000,100,buy
6,0,1,3190000,24,buy
7,0,1,3150000,18,buy
8,0,1,3130000,2,buy
9,0,1,3110000,5,buy


{'books': 1,
 'bytes': 9349472,
 'checksum_checks': 0,
 'checksum_failures': 0,
 'clock_ns': 14547228549320,
 'complete': False,
 'messages': 326777,
 'symbol': 'AAPL',
 'synchronized': True}

Close this session before starting the next source.

In [7]:
terminal.close()

## Bitfinex public feed

In [8]:
definition = bitfinex.protocol()
source = lm.Source.websocket("wss://api-pub.bitfinex.com/ws/2")
started = perf_counter()
adapter = CustomAdapter(
    definition, source, name='Bitfinex R0', symbol='BTCUSD', scope=['BTCUSD'],
    mode='live', level="l3", timezone='UTC',
)
print(f"Construction (includes compilation or cache reuse): {(perf_counter() - started) * 1000:.3f} ms")
terminal = server_context(adapters=[adapter], port=0)
display(HTML(f'<a href="{terminal.url}" target="_blank">Open order-book terminal</a>'))

Construction (includes compilation or cache reuse): 8.599 ms


In [9]:
display(pd.DataFrame(wait_for_book(adapter, 'BTCUSD')).head(12))
adapter.status()

,hidden,orders,price,quantity,side
0,0,1,7896800000000,22000,buy
1,0,1,7896700000000,542326,buy
2,0,1,7896300000000,22000,buy
3,0,2,7896000000000,5496581,buy
4,0,2,7895900000000,3695333,buy
5,0,1,7895800000000,22000,buy
6,0,2,7895600000000,2221044,buy
7,0,3,7895200000000,7360633,buy
8,0,1,7895100000000,27630,buy
9,0,1,7894700000000,63000,buy


{'books': 1,
 'bytes': 32053,
 'checksum_checks': 1,
 'checksum_failures': 0,
 'clock_ns': 1788932755615900750,
 'complete': False,
 'messages': 260,
 'symbol': 'BTCUSD',
 'synchronized': True}

Close this session before starting the next source.

In [10]:
terminal.close()